In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama

In [ ]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict): 
    messages: Annotated[list[BaseMessage], add_messages]
    

In [6]:
llm = ChatOllama(model='llama3.2')

In [7]:
def chat_node(state: ChatState): 
    #take a user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages) 

    # response store state
    return {"messages": response}

In [11]:
graph = StateGraph(ChatState)

#add nodes
graph.add_node("chat_node", chat_node)

#add edges 
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)
workflow = graph.compile()

In [14]:
inital_state = {
    "messages": [HumanMessage(content="What is the capital of Nepal?")]
}
workflow.invoke(inital_state)['messages'][-1].content

'The capital of Nepal is Kathmandu.'

In [ ]:
while True: 
    user_message = input("Type here: ")
    print("User: ", user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye', '0']:
        break

    response = workflow.invoke({'messages': [HumanMessage(content=user_message)]})

    print('AI', response['messages'][-1].content)


    